In [ ]:
!pip install pyethnicity

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.8/38.8 MB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.5/300.5 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.3 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import os
import pyethnicity

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
file_path = "/Citywide_Payroll_Data_(Fiscal_Year)_2023.csv"  # replace with your file path
base_path = "/content/drive/MyDrive/Fall 2025/Responsible AI/RAI Project/Project Implementation"
df_bisg = pd.read_csv(base_path + file_path)

/tmp/ipython-input-1841873807.py:3: DtypeWarning: Columns (12,14) have mixed types. Specify dtype option on import or set low_memory=False.
  df_bisg = pd.read_csv(base_path + file_path)


In [ ]:
df_bisg.columns = df_bisg.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('-', '_')

df_bisg.rename(columns={
    'mid_init': 'middle_name',
    'last_name': 'last_name',
    'first_name': 'first_name'
}, inplace=True)


In [ ]:
print(list(df_bisg.columns))

['fiscal_year', 'payroll_number', 'agency_name', 'last_name', 'first_name', 'middle_name', 'agency_start_date', 'work_location_borough', 'title_description', 'leave_status_as_of_june_30', 'base_salary', 'pay_basis', 'regular_hours', 'regular_gross_paid', 'ot_hours', 'total_ot_paid', 'total_other_pay']


In [ ]:
# bisg_cols = ['first_name', 'middle_name', 'last_name', 'work_location_borough', 'base_salary']
# df_bisg = df[bisg_cols].copy()
print(df_bisg.size, len(df_bisg))

9569266 562898


In [ ]:
# for col in bisg_cols:
#     df_bisg[col] = df_bisg[col].astype(str).str.strip().str.title()

# Replace empty strings or 'nan' with actual NaN
df_bisg.replace({'': pd.NA, 'Nan': pd.NA, 'nan': pd.NA}, inplace=True)

df_bisg.dropna(subset=['first_name', 'last_name', 'base_salary'], inplace=True)
df_bisg.reset_index(drop=True, inplace=True)
len(df_bisg)

562468

In [ ]:
bisg_cols = ['first_name', 'middle_name', 'last_name', 'work_location_borough', 'base_salary']

# Dropping rows with NaN
# df_bisg.dropna(subset=bisg_cols, inplace=True)

#dropping rows with white spaces
df_bisg = df_bisg[~df_bisg[bisg_cols].apply(lambda x: x.str.strip() == '').any(axis=1)]
df_bisg.reset_index(drop=True, inplace=True)
print(len(df_bisg))

#dropping rows for which the boroughs are other or washington dc
# df_bisg = df_bisg[~df_bisg['work_location_borough'].isin(['OTHER', 'WASHINGTON DC'])]

562468


In [ ]:
print(df_bisg.head())
print(f"Rows remaining after cleaning: {len(df_bisg)}")


   fiscal_year  payroll_number                agency_name      last_name  \
0         2024              67  ADMIN FOR CHILDREN'S SVCS          FINCH   
1         2024              67  ADMIN FOR CHILDREN'S SVCS        MCGRATH   
2         2024              67  ADMIN FOR CHILDREN'S SVCS          AGEDA   
3         2024              67  ADMIN FOR CHILDREN'S SVCS  HOWARD-COOPER   
4         2024              67  ADMIN FOR CHILDREN'S SVCS        STEVENS   

  first_name middle_name agency_start_date work_location_borough  \
0     ROBERT           J        07/06/1997             MANHATTAN   
1    MATTHEW           M        04/15/2013              BROOKLYN   
2     ADRIAN           A        03/28/2022              BROOKLYN   
3      ELLEN         NaN        03/12/2018             MANHATTAN   
4    DAMARIS           O        05/16/2016             MANHATTAN   

              title_description leave_status_as_of_june_30  base_salary  \
0  ADMINISTRATIVE STAFF ANALYST                     CEASED 

In [ ]:
# --------------------------
# 6. Ready for BISG
# --------------------------

In [ ]:
# column_data = df_bisg['work_location_borough']

# unique_values = column_data.unique()

# print(unique_values)

['MANHATTAN' 'BROOKLYN' 'QUEENS' 'BRONX' 'RICHMOND' 'WESTCHESTER' 'NASSAU'
 'DELAWARE' 'SULLIVAN' 'ORANGE' 'ULSTER' 'SCHOHARIE' 'ALBANY' 'PUTNAM'
 'DUTCHESS' 'GREENE']


In [ ]:
ny_zip_map = {
        "MANHATTAN": [
        10001, 10002, 10003, 10004, 10005, 10006, 10007, 10009, 10010, 10011,
        10012, 10013, 10014, 10016, 10017, 10018, 10019, 10021, 10022, 10023,
        10024, 10025, 10026, 10027, 10028, 10029, 10030, 10031, 10032, 10033,
        10034, 10035, 10036, 10037, 10038, 10039, 10040, 10044, 10069, 10103,
        10119, 10128, 10162, 10165, 10170, 10173, 10199, 10279, 10280, 10282
    ],
   "BROOKLYN": [
        11201, 11206, 11207, 11208, 11209, 11202, 11203, 11204, 11205, 11210,
        11211, 11212, 11213, 11218, 11219, 11220, 11221, 11222, 11223, 11224,
        11225, 11214, 11215, 11216, 11217, 11226, 11228, 11229, 11230, 11235,
        11236, 11237, 11238, 11245, 11247, 11249, 11256, 11231, 11232, 11233,
        11234, 11239, 11241, 11242, 11243, 11251, 11252
    ],
   "BRONX": [
        10451, 10452, 10453, 10454, 10455, 10456, 10457, 10458, 10459, 10460,
        10461, 10462, 10463, 10464, 10465, 10466, 10467, 10468, 10469, 10470,
        10471, 10472, 10473, 10474, 10475
    ],
        "QUEENS": sorted(list(set([
        11004, 11005, 11101, 11102, 11103, 11104, 11105, 11106, 11109,
        11351, 11352, 11354, 11355, 11356, 11357, 11358, 11359, 11360, 11361,
        11362, 11363, 11364, 11365, 11366, 11367, 11368, 11369, 11370, 11371,
        11372, 11373, 11374, 11375, 11377, 11378, 11379, 11380, 11385, 11386,
        11411, 11412, 11413, 11414, 11415, 11416, 11417, 11418, 11419, 11420,
        11421, 11422, 11423, 11424, 11426, 11427, 11428, 11429, 11430, 11431,
        11432, 11433, 11434, 11435, 11436, 11690, 11691, 11692, 11693, 11694,
        11695, 11697
    ]))),
    "RICHMOND": [
        10301, 10302, 10303, 10304, 10305, 10306, 10307, 10308,
        10309, 10310, 10311, 10312, 10313, 10314
    ],
    "ALBANY": [
        12201, 12202, 12203, 12204, 12205, 12206, 12207, 12208, 12209,
        12210, 12211, 12212, 12220, 12223, 12224, 12225, 12226, 12227,
        12247, 12260
    ],
    "WESTCHESTER": [
        10501, 10502, 10503, 10504, 10505, 10506, 10507, 10510, 10511, 10514,
        10517, 10518, 10519, 10520, 10521, 10522, 10523, 10526, 10527, 10528,
        10530, 10532, 10533, 10535, 10536, 10538, 10540, 10543, 10545, 10546,
        10547, 10548, 10549, 10550, 10551, 10552, 10553, 10560, 10562, 10566,
        10567, 10570, 10573, 10576, 10577, 10578, 10580, 10583, 10587, 10588,
        10589, 10590, 10591, 10594, 10595, 10596, 10597
    ],
    "PUTNAM": [
        10509, 10512, 10516, 10524, 10537, 10541, 10542, 10579
    ],
    "NASSAU": sorted(list(set([
        11001, 11002, 11003, 11004, 11005, 11010, 11020, 11021, 11022, 11023,
        11024, 11026, 11027, 11030, 11040, 11042, 11050, 11096, 11501, 11507,
        11509, 11510, 11514, 11516, 11518, 11520, 11530, 11531, 11542, 11545,
        11547, 11548, 11550, 11551, 11552, 11553, 11554, 11557, 11558, 11559,
        11560, 11561, 11563, 11565, 11566, 11568, 11569, 11570, 11571, 11572,
        11575, 11576, 11577, 11579, 11580, 11581, 11582, 11596, 11599, 11709,
        11710, 11714, 11732, 11735, 11753, 11756, 11758, 11762, 11765, 11771,
        11783, 11791, 11793, 11801, 11802, 11803, 11804
    ]))),
     "DELAWARE": sorted(list(set([
        12167, 12406, 12421, 12430, 12434, 12438, 12455, 12459, 12474,
        13731, 13739, 13740, 13750, 13751, 13752, 13753, 13755, 13756,
        13757, 13774, 13775, 13782, 13783, 13786, 13788, 13804, 13806,
        13837, 13838, 13839, 13842, 13846, 13847, 13856, 13860
    ]))),
     "SULLIVAN": sorted(list(set([
        12701, 12719, 12720, 12721, 12722, 12723, 12724, 12725, 12726, 12727,
        12732, 12733, 12734, 12736, 12737, 12738, 12740, 12741, 12742, 12743,
        12745, 12747, 12748, 12749, 12750, 12751, 12752, 12754, 12758, 12759,
        12760, 12762, 12763, 12764, 12765, 12766, 12767, 12768, 12769, 12770,
        12775, 12776, 12777, 12778, 12779, 12781, 12783, 12784, 12785
    ]))),
    "ORANGE": sorted(list(set([
        10910, 10912, 10914, 10915, 10916, 10917, 10918, 10919, 10921, 10922,
        10924, 10925, 10926, 10928, 10930, 10932, 10933, 10940, 10941, 10950,
        10953, 10958, 10959, 10963, 10969, 10973, 10975, 10979, 10981, 10985,
        10987, 10988, 10990, 10992, 10996, 10997, 12518, 12520, 12543, 12549,
        12550, 12551, 12552, 12553, 12555, 12566, 12575, 12577, 12584, 12586,
        12729, 12739, 12746, 12771, 12780
    ]))),
    "ULSTER": sorted(list(set([
        12401, 12402, 12404, 12409, 12410, 12411, 12412, 12416, 12417, 12419,
        12420, 12428, 12429, 12432, 12433, 12435, 12440, 12441, 12443, 12446,
        12448, 12449, 12453, 12456, 12457, 12458, 12461, 12464, 12465, 12466,
        12471, 12472, 12475, 12477, 12480, 12481, 12483, 12484, 12486, 12487,
        12489, 12490, 12491, 12493, 12515, 12525, 12528, 12542, 12547, 12548,
        12561, 12568, 12588, 12589, 12782
    ]))),
    "SCHOHARIE": sorted(list(set([
        12031, 12035, 12036, 12043, 12071, 12073, 12076, 12092, 12093,
        12122, 12131, 12149, 12157, 12160, 12175, 12187, 12194, 13459
    ]))),
    "DUTCHESS": sorted(list(set([
        12501, 12504, 12506, 12507, 12508, 12510, 12511, 12512, 12514, 12522,
        12524, 12527, 12531, 12533, 12537, 12538, 12540, 12545, 12546, 12564,
        12567, 12569, 12570, 12571, 12572, 12574, 12578, 12580, 12581, 12582,
        12583, 12585, 12590, 12592, 12601, 12602, 12603
    ]))),
    'GREENE': [13778]
}

In [ ]:
# crosswalk_path = os.path.join(base_path, 'ZIP Code to ZCTA Crosswalk.xlsx')
# crosswalk_df = pd.read_excel(crosswalk_path)

# zip_to_zcta = crosswalk_df.set_index("ZIP_CODE")["zcta"].to_dict()

# place_to_zcta = {}

# for place, zip_list in ny_zip_map.items():
#     zcta_list = [int(zip_to_zcta[z]) for z in zip_list if z in zip_to_zcta and pd.notna(zip_to_zcta[z])]
#     place_to_zcta[place] = zcta_list

# print(place_to_zcta["MANHATTAN"])

[10001, 10002, 10003, 10004, 10005, 10006, 10007, 10009, 10010, 10011, 10012, 10013, 10014, 10016, 10017, 10018, 10019, 10021, 10022, 10023, 10024, 10025, 10026, 10027, 10028, 10029, 10030, 10031, 10032, 10033, 10034, 10035, 10036, 10037, 10038, 10039, 10040, 10044, 10069, 10103, 10119, 10128, 10162, 10165, 10170, 10173, 10199, 10279, 10280, 10282]


In [ ]:
# df_bisg['borough_zctas'] = df_bisg['work_location_borough'].map(place_to_zcta)

# df_bisg['borough_zctas'] = df_bisg['borough_zctas'].apply(lambda x: x if isinstance(x, list) else [])

In [ ]:
# df_bisg.to_csv("cleaned_bisg_with_zctas.csv", index=False)

In [ ]:
# df_bisg.size
# print(df_bisg.head())

  first_name middle_name last_name work_location_borough  base_salary  \
0     ROBERT           J     FINCH             MANHATTAN  $149,836.00   
1    MATTHEW           M   MCGRATH              BROOKLYN  $139,472.00   
2     ADRIAN           A     AGEDA              BROOKLYN      $555.84   
3    DAMARIS           O   STEVENS             MANHATTAN   $67,899.00   
4    DARLESE           R     SMITH             MANHATTAN  $115,431.00   

                                       borough_zctas  
0  [10001, 10002, 10003, 10004, 10005, 10006, 100...  
1  [11201, 11206, 11207, 11208, 11209, 11201, 112...  
2  [11201, 11206, 11207, 11208, 11209, 11201, 112...  
3  [10001, 10002, 10003, 10004, 10005, 10006, 100...  
4  [10001, 10002, 10003, 10004, 10005, 10006, 100...  


In [ ]:

count = 0;
print("Conducting bifsg on df_bisg:")
for index, row in df_bisg.iterrows():
    first_name = row['first_name']
    last_name = row['last_name']
    # zcta_value = row['borough_zctas']


    count +=1

    if count % 1000 == 0:
      print(f"\nProcessing record {index+1}") #: First Name='{first_name}', Last Name='{last_name}', ZCTA='{zcta_value}'")
    try:

        # result = pyethnicity.bifsg(
        #     first_name,
        #     last_name,
        #     zcta_value,
        #     "zcta"
        # )

        result = pyethnicity.predict_race_fl(first_name,last_name)

        print(result)

        # mean_probs = result[['asian', 'black', 'white', 'hispanic']].mean()
        df_bisg.loc[row.name, 'prob_white'] = result['white'].item()
        df_bisg.loc[row.name, 'prob_black'] = result['black'].item()
        df_bisg.loc[row.name, 'prob_hisp'] = result['hispanic'].item()
        df_bisg.loc[row.name, 'prob_asian'] = result['asian'].item()

        # print(row['race_code'])
    except Exception as e:
        print(f"Error conducting bifsg for this record: {e}")
print(df_bisg.head() )
df_bisg.to_csv('bisg_done_sample_fl_nyc.csv', index=False)


Conducting bifsg on df_bisg:
*************** EP Error ***************
EP Error /onnxruntime_src/onnxruntime/python/onnxruntime_pybind_state.cc:560 void onnxruntime::python::RegisterTensorRTPluginsAsCustomOps(PySessionOptions&, const onnxruntime::ProviderOptions&) Please install TensorRT libraries as mentioned in the GPU requirements page, make sure they're in the PATH or LD_LIBRARY_PATH, and that your GPU is supported.
 when using ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
Falling back to ['CPUExecutionProvider'] and retrying.
****************************************


100%|██████████| 1/1 [00:00<00:00, 13.57it/s]


shape: (1, 6)
┌────────────┬───────────┬──────────┬─────────┬──────────┬──────────┐
│ first_name ┆ last_name ┆ asian    ┆ black   ┆ hispanic ┆ white    │
│ ---        ┆ ---       ┆ ---      ┆ ---     ┆ ---      ┆ ---      │
│ str        ┆ str       ┆ f32      ┆ f32     ┆ f32      ┆ f32      │
╞════════════╪═══════════╪══════════╪═════════╪══════════╪══════════╡
│ ROBERT     ┆ FINCH     ┆ 0.013039 ┆ 0.21077 ┆ 0.008956 ┆ 0.767235 │
└────────────┴───────────┴──────────┴─────────┴──────────┴──────────┘


100%|██████████| 1/1 [00:00<00:00, 13.93it/s]


shape: (1, 6)
┌────────────┬───────────┬──────────┬──────────┬──────────┬──────────┐
│ first_name ┆ last_name ┆ asian    ┆ black    ┆ hispanic ┆ white    │
│ ---        ┆ ---       ┆ ---      ┆ ---      ┆ ---      ┆ ---      │
│ str        ┆ str       ┆ f32      ┆ f32      ┆ f32      ┆ f32      │
╞════════════╪═══════════╪══════════╪══════════╪══════════╪══════════╡
│ MATTHEW    ┆ MCGRATH   ┆ 0.053359 ┆ 0.008118 ┆ 0.013312 ┆ 0.925212 │
└────────────┴───────────┴──────────┴──────────┴──────────┴──────────┘


...

shape: (1, 6)
┌────────────┬───────────┬──────────┬──────────┬──────────┬──────────┐
│ first_name ┆ last_name ┆ asian    ┆ black    ┆ hispanic ┆ white    │
│ ---        ┆ ---       ┆ ---      ┆ ---      ┆ ---      ┆ ---      │
│ str        ┆ str       ┆ f32      ┆ f32      ┆ f32      ┆ f32      │
╞════════════╪═══════════╪══════════╪══════════╪══════════╪══════════╡
│ ADRIAN     ┆ AGEDA     ┆ 0.036087 ┆ 0.081896 ┆ 0.876541 ┆ 0.005476 │
└────────────┴───────────┴──────────┴──────────┴──────────┴──────────┘


100%|██████████| 1/1 [00:00<00:00, 11.56it/s]


shape: (1, 6)
┌────────────┬───────────────┬──────────┬──────────┬──────────┬──────────┐
│ first_name ┆ last_name     ┆ asian    ┆ black    ┆ hispanic ┆ white    │
│ ---        ┆ ---           ┆ ---      ┆ ---      ┆ ---      ┆ ---      │
│ str        ┆ str           ┆ f32      ┆ f32      ┆ f32      ┆ f32      │
╞════════════╪═══════════════╪══════════╪══════════╪══════════╪══════════╡
│ ELLEN      ┆ HOWARD-COOPER ┆ 0.085175 ┆ 0.145702 ┆ 0.014814 ┆ 0.754309 │
└────────────┴───────────────┴──────────┴──────────┴──────────┴──────────┘


100%|██████████| 1/1 [00:00<00:00, 12.84it/s]


shape: (1, 6)
┌────────────┬───────────┬──────────┬──────────┬──────────┬──────────┐
│ first_name ┆ last_name ┆ asian    ┆ black    ┆ hispanic ┆ white    │
│ ---        ┆ ---       ┆ ---      ┆ ---      ┆ ---      ┆ ---      │
│ str        ┆ str       ┆ f32      ┆ f32      ┆ f32      ┆ f32      │
╞════════════╪═══════════╪══════════╪══════════╪══════════╪══════════╡
│ DAMARIS    ┆ STEVENS   ┆ 0.021064 ┆ 0.439301 ┆ 0.3888   ┆ 0.150835 │
└────────────┴───────────┴──────────┴──────────┴──────────┴──────────┘
   fiscal_year  payroll_number                agency_name      last_name  \
0         2024              67  ADMIN FOR CHILDREN'S SVCS          FINCH   
1         2024              67  ADMIN FOR CHILDREN'S SVCS        MCGRATH   
2         2024              67  ADMIN FOR CHILDREN'S SVCS          AGEDA   
3         2024              67  ADMIN FOR CHILDREN'S SVCS  HOWARD-COOPER   
4         2024              67  ADMIN FOR CHILDREN'S SVCS        STEVENS   

  first_name middle_name agency_